In [66]:
import os
import pickle
import base64
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from io import BytesIO
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

In [62]:
def fig_to_base64(fig):
    buf = BytesIO()
    fig.savefig(buf, format='png', bbox_inches='tight', dpi=150)
    buf.seek(0)
    b64 = base64.b64encode(buf.read()).decode('utf-8')
    plt.close(fig)
    return b64

In [34]:
root_path = "/Users/kautsarg/Documents/Final Project/Run Data/trial test data/"
sample_name = "D20250808_E00_C00_F4500KHz_U_Sample_7"
exp_path = os.path.join(root_path, sample_name)

result_path = os.path.join(exp_path, "classification_performances_with_proba.pkl")
with open(result_path, 'rb') as f:
    result_data = pickle.load(f)

curve_path = os.path.join(exp_path, "curve_for_training_latest.pkl")
with open(curve_path, 'rb') as f:
    curve_data = pickle.load(f)

dataset_name = curve_data["dataset_name"]
dataset_name_clean = [name.replace("_", " ").title() for name in  dataset_name]
dataset = curve_data["dataset"]
kinetic_features = curve_data["kinetic_features"]
Y_well = curve_data["Y_well"]

In [76]:
dataset_name

['ori_curves',
 'ori_curves_avg',
 'original_fitted_full',
 'original_fitted_stretched',
 'cleaned_std_fitted_full',
 'cleaned_std_fitted_stretched',
 'cleaned_lowest_fitted_full',
 'cleaned_lowest_fitted_stretched']

In [68]:
method = "Reference"
outlier_filters = ["Baseline_None", "msc_label_msc_linear_0.001", "msc_label_msc_baseline_0.001", "amf_label_amf_important", "amf_label_amf_send_5", "knn_top_0.9", "knn_top_0.95", "cnn_ae_label_95", "cnn_ae_label_elbow"]
dataset_name_filters = ['Ori Curves', 'Original Fitted Full', 'Cleaned Std Fitted Full', 'Cleaned Std Fitted Stretched']

reports_dir = os.path.join(exp_path, "evaluation_reports")
os.makedirs(reports_dir, exist_ok=True)

encoder = LabelEncoder()
y_full = encoder.fit_transform(Y_well)

for i, ds_name in enumerate(dataset_name_clean):
    if ds_name in dataset_name_filters:
        curves = dataset[i]
        
        for outlier_filter in outlier_filters:
            result_filter = None if outlier_filter == "Baseline_None" else outlier_filter
            dataset_filter = outlier_filter
            
            if result_filter not in result_data[ds_name][method]:
                print(f"[SKIP] Results for {outlier_filter} not found in {ds_name} dict.")
                continue
            
            # 1. APPLY INITIAL FILTER MASK
            if dataset_filter == "Baseline_None":
                mask = np.ones(len(y_full), dtype=bool)
            else:
                # Ensure we handle NaN correctly as in model_utils.py
                mask = (kinetic_features[i][dataset_filter] == 1).fillna(False).values
                
            inlier_curves = curves[mask]
            y_true_encoded = y_full[mask]

            # 2. IDENTICAL RARE CLASS FILTERING (Lines 105-112 in model_utils.py)
            unique_classes, class_counts = np.unique(y_true_encoded, return_counts=True)
            rare_classes = unique_classes[class_counts < 2]

            if len(rare_classes) > 0:
                valid_class_mask = ~np.isin(y_true_encoded, rare_classes)
                inlier_curves = inlier_curves[valid_class_mask]
                y_true_encoded = y_true_encoded[valid_class_mask]

            n_classes = len(np.unique(y_true_encoded))
            if n_classes < 2 or len(y_true_encoded) < 2 * n_classes:
                print(f"[SKIP] Not enough classes/samples to split {ds_name} - {outlier_filter}")
                continue

            # 3. IDENTICAL STRATIFIED SHUFFLE SPLIT
            calculated_test_size = max(int(len(y_true_encoded) * 0.10), n_classes)
            sss = StratifiedShuffleSplit(n_splits=1, test_size=calculated_test_size, random_state=0)
            splits = list(sss.split(inlier_curves, y_true_encoded))
            
            test_idx = splits[0][1]
            tested_curves = inlier_curves[test_idx]
            
            # 4. FETCH PREDICTIONS & DECODE
            y_preds_encoded = np.array(result_data[ds_name][method][result_filter]["y_preds_AC_"][0])
            y_trues_encoded = np.array(result_data[ds_name][method][result_filter]["y_trues_"][0])
            
            # Check for exact alignment
            if len(tested_curves) != len(y_preds_encoded):
                print(f"[WARNING] Size mismatch on {ds_name} | {outlier_filter}. Curves: {len(tested_curves)}, Preds: {len(y_preds_encoded)}")
                continue
            
            # Convert integer labels back to real string names for plotting
            y_preds = encoder.inverse_transform(y_preds_encoded)
            y_trues = encoder.inverse_transform(y_trues_encoded)
            display_classes = np.unique(y_trues)
            
            # ==========================================
            # 5. Calculate Metrics & CM
            # ==========================================
            cm = confusion_matrix(y_trues, y_preds, labels=display_classes)
            
            # Class Accuracy: Correct / Total True
            class_totals = cm.sum(axis=1)
            per_class_acc = np.divide(cm.diagonal(), class_totals, out=np.zeros_like(cm.diagonal(), dtype=float), where=class_totals!=0)
            
            acc = accuracy_score(y_trues, y_preds)
            prec_per_class = precision_score(y_trues, y_preds, average=None, labels=display_classes, zero_division=0)
            rec_per_class = recall_score(y_trues, y_preds, average=None, labels=display_classes, zero_division=0)
            f1_per_class = f1_score(y_trues, y_preds, average=None, labels=display_classes, zero_division=0)
            
            metrics_data = {
                "Class (Well)": display_classes,
                "Accuracy": np.round(per_class_acc, 4),
                "Precision": np.round(prec_per_class, 4),
                "Recall": np.round(rec_per_class, 4),
                "F1-Score": np.round(f1_per_class, 4),
                "Support": class_totals
            }
            metrics_df = pd.DataFrame(metrics_data)
            
            metrics_df.loc["Total/Macro"] = [
                "Macro Avg", 
                np.round(np.mean(per_class_acc), 4),
                np.round(np.mean(prec_per_class), 4), 
                np.round(np.mean(rec_per_class), 4), 
                np.round(np.mean(f1_per_class), 4),
                len(y_trues)
            ]
            
            # ==========================================
            # 6. HTML Initialization & Styling
            # ==========================================
            html = f"""
            <html>
            <head>
                <title>{ds_name} - {outlier_filter}</title>
                <style>
                    body {{ font-family: 'Segoe UI', Arial, sans-serif; background-color: #e9ecef; padding: 20px; color: #333; }}
                    .header {{ text-align: center; margin-bottom: 30px; }}
                    .grid-container {{ display: grid; grid-template-columns: 1fr 1fr; gap: 20px; align-items: start; margin-bottom: 30px; }}
                    .card {{ background: white; padding: 20px; border-radius: 10px; box-shadow: 0 4px 6px rgba(0,0,0,0.1); }}
                    h1 {{ color: #2c3e50; margin-bottom: 5px; }}
                    h2 {{ color: #34495e; font-size: 1.2rem; border-bottom: 2px solid #eee; padding-bottom: 10px; margin-top: 0; }}
                    h3 {{ color: #16a085; text-align: center; }}
                    .table-container {{ width: 100%; overflow-x: auto; }}
                    table {{ border-collapse: collapse; width: 100%; margin: auto; font-size: 0.9rem; }}
                    th, td {{ border: 1px solid #ddd; padding: 10px; text-align: center; }}
                    th {{ background-color: #34495e; color: white; }}
                    tr:nth-child(even) {{ background-color: #f8f9fa; }}
                    tr:last-child {{ font-weight: bold; background-color: #e2e6ea; }}
                    img {{ max-width: 100%; height: auto; border-radius: 4px; display: block; margin: auto; }}
                    .section-title {{ text-align: center; margin: 40px 0 20px 0; color: #2c3e50; }}
                </style>
            </head>
            <body>
                <div class="header">
                    <h1>Performance Report</h1>
                    <p><b>Dataset:</b> {ds_name} &nbsp;|&nbsp; <b>Outlier Filter:</b> {outlier_filter}</p>
                </div>
            """
            
            # ==========================================
            # Row 1: Metrics Table (Left) & CM (Right)
            # ==========================================
            html += "<div class='grid-container'>"
            
            html += f"""
                <div class='card'>
                    <h2>Classification Metrics</h2>
                    <h3>Overall Accuracy: {acc:.2%}</h3>
                    <div class='table-container'>{metrics_df.to_html(index=False, classes="table")}</div>
                </div>
            """
            
            fig_cm, ax_cm = plt.subplots(figsize=(6, 5))
            sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=display_classes, yticklabels=display_classes, ax=ax_cm)
            ax_cm.set_xlabel('Predicted Label', fontweight='bold')
            ax_cm.set_ylabel('True Label', fontweight='bold')
            plt.tight_layout()
            
            cm_b64 = fig_to_base64(fig_cm)
            html += f"""
                <div class='card'>
                    <h2>Confusion Matrix</h2>
                    <img src='data:image/png;base64,{cm_b64}'>
                </div>
            </div>
            """
            
            # ==========================================
            # Subsequent Rows: Plot Grids (Label 0 | Label 1)
            # ==========================================
            html += "<h2 class='section-title'>Prediction Visualizations by Class</h2>"
            html += "<div class='grid-container'>"
            
            for cls, cls_acc in zip(display_classes, per_class_acc):
                cls_mask = (y_trues == cls)
                correct_mask = cls_mask & (y_trues == y_preds)
                incorrect_mask = cls_mask & (y_trues != y_preds)
                
                fig_curves, axes = plt.subplots(1, 2, figsize=(10, 4.5))
                fig_curves.suptitle(f"Label {cls} | Accuracy: {cls_acc:.2%}", fontweight='bold', fontsize=14, color='#2c3e50')
                
                # Correct Predictions
                if correct_mask.sum() > 0:
                    axes[0].plot(tested_curves[correct_mask].T, color='green', alpha=0.3, rasterized=True)
                axes[0].set_title(f"Correct (n={correct_mask.sum()})", fontweight='bold', color='green')
                axes[0].set_ylabel("Fluorescence")
                axes[0].set_xlabel("Time/Cycle")
                
                # Incorrect Predictions
                if incorrect_mask.sum() > 0:
                    axes[1].plot(tested_curves[incorrect_mask].T, color='red', alpha=0.5, rasterized=True)
                axes[1].set_title(f"Incorrect (n={incorrect_mask.sum()})", fontweight='bold', color='red')
                axes[1].set_xlabel("Time/Cycle")
                
                plt.tight_layout()
                curves_b64 = fig_to_base64(fig_curves)
                
                html += f"""
                <div class='card'>
                    <img src='data:image/png;base64,{curves_b64}'>
                </div>
                """
            
            # ==========================================
            # Finalize HTML and Save
            # ==========================================
            html += """
                </div>
            </body>
            </html>
            """
            
            safe_ds_name = ds_name.replace(" ", "_").lower()
            safe_filter_name = outlier_filter if outlier_filter else "baseline_none"
            file_name = f"{safe_ds_name}_{safe_filter_name}.html"
            save_path = os.path.join(reports_dir, file_name)
            
            with open(save_path, "w") as f:
                f.write(html)
                
            print(f"  -> Generated Report: {file_name}")

print("-" * 100)
print(f"[*] All reports successfully generated in: {reports_dir}")

[WARNING] Size mismatch on Ori Curves | amf_label_amf_important. Curves: 1044, Preds: 1038
